In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

gemma_270m = 'google/gemma-3-270m'
local_path = './models/gemma_function_model_exp3'
tokenizer = AutoTokenizer.from_pretrained(gemma_270m)
model = AutoModelForCausalLM.from_pretrained(local_path, device_map="cpu")

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ToolingGemma:
    def __init__(self, system_instructions):
        self.chat_history = ''
        self.model = AutoModelForCausalLM.from_pretrained('./models/gemma_function_model_exp3', device_map="cpu")
        self.tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-270m')
        self.system_instructions = system_instructions
        self.stop_word = "<|endoftext|>"
    
    def generate(self, user_query):
        user_query = 'USER: ' + user_query
        if self.chat_history:
            prompt = self.chat_history + '\n' + user_query + '\n' + 'ASSISTANT:'
        else:
            prompt = self.system_instructions + '\n\n' + user_query + '\n' + 'ASSISTANT:'
        
        input_ids = self.tokenizer(prompt, return_tensors="pt")
        
        agent_response = self.model.generate(
            **input_ids, 
            generation_config=GenerationConfig.from_dict({"max_new_tokens": 1000}),
            stop_strings=[self.stop_word],
            tokenizer=self.tokenizer,
        )

        decoded_agent_response = tokenizer.decode(agent_response[0])
        self.chat_history = decoded_agent_response

        return decoded_agent_response

In [24]:
system_instructions = '''
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_discount",
    "description": "Calculate the discounted price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "original_price": {
                "type": "number",
                "description": "The original price of the product"
            },
            "discount_percentage": {
                "type": "number",
                "description": "The discount percentage"
            }
        },
        "required": [
            "original_price",
            "discount_percentage"
        ]
    }
}
'''

tooling_gemma_model = ToolingGemma(system_instructions=system_instructions)

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 19483.05it/s]


In [25]:
print(tooling_gemma_model.generate('Can you please book a flight for me from New York to London?'))

<bos>
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_discount",
    "description": "Calculate the discounted price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "original_price": {
                "type": "number",
                "description": "The original price of the product"
            },
            "discount_percentage": {
                "type": "number",
                "description": "The discount percentage"
            }
        },
        "required": [
            "original_price",
            "discount_percentage"
        ]
    }
}


USER: Can you please book a flight for me from New York to London?
ASSISTANT: I'm sorry, but I'm unable to assist with booking flights. My current capabilities are limited to calculating discounted prices based on original price and discount percentage. If you need help with that, feel free to ask! <|endoftext|>


In [26]:
print(tooling_gemma_model.generate('Calculate the discounted price for 100 dollars at a discount of 30%'))

<bos><bos>
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_discount",
    "description": "Calculate the discounted price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "original_price": {
                "type": "number",
                "description": "The original price of the product"
            },
            "discount_percentage": {
                "type": "number",
                "description": "The discount percentage"
            }
        },
        "required": [
            "original_price",
            "discount_percentage"
        ]
    }
}


USER: Can you please book a flight for me from New York to London?
ASSISTANT: I'm sorry, but I'm unable to assist with booking flights. My current capabilities are limited to calculating discounted prices based on original price and discount percentage. If you need help with that, feel free to ask! <|endoftext

In [5]:
class ToolingGemma:
    def __init__(self, system_instructions):
        self.chat_history = ''
        self.model = AutoModelForCausalLM.from_pretrained('SauravP97/tooling-gemma-270M-inst', device_map="cpu")
        self.tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-270m')
        self.system_instructions = system_instructions
        self.stop_word = "<|endoftext|>"

    def generate(self, user_query):
        user_query = 'USER: ' + user_query
        if self.chat_history:
            prompt = self.chat_history + '\n' + user_query + '\n' + 'ASSISTANT:'
        else:
            prompt = self.system_instructions + '\n\n' + user_query + '\n' + 'ASSISTANT:'

        input_ids = self.tokenizer(prompt, return_tensors="pt")

        agent_response = self.model.generate(
            **input_ids, 
            generation_config=GenerationConfig.from_dict({"max_new_tokens": 1000}),
            stop_strings=[self.stop_word],
            tokenizer=self.tokenizer,
        )

        decoded_agent_response = self.tokenizer.decode(agent_response[0])
        self.chat_history = decoded_agent_response

        return decoded_agent_response

system_instructions = '''
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_discount",
    "description": "Calculate the discounted price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "original_price": {
                "type": "number",
                "description": "The original price of the product"
            },
            "discount_percentage": {
                "type": "number",
                "description": "The discount percentage"
            }
        },
        "required": [
            "original_price",
            "discount_percentage"
        ]
    }
}
'''

tooling_gemma_model = ToolingGemma(system_instructions=system_instructions)

agent_response = tooling_gemma_model.generate('Can you please book a flight for me from New York to London?')


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 16560.25it/s]


In [7]:
print(agent_response)

<bos>
SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "calculate_discount",
    "description": "Calculate the discounted price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "original_price": {
                "type": "number",
                "description": "The original price of the product"
            },
            "discount_percentage": {
                "type": "number",
                "description": "The discount percentage"
            }
        },
        "required": [
            "original_price",
            "discount_percentage"
        ]
    }
}


USER: Can you please book a flight for me from New York to London?
ASSISTANT: I'm sorry, but I'm unable to assist with that. My current capabilities allow me to calculate discounts on products, but I don't have the ability to book flights or perform other external tasks. <|endoftext|>
